# Cours 2 "Traitement informatique des données" - Statistiques appliquées aux sciences humaines
### Louis Maritaud
### louis.maritaud@unilim.fr

*On reprend le même jeu de données que le cours sur les graphiques : la population des communes françaises depuis 1876 (source INSEE). Placez `pop_hist.csv` dans le même dossier que ce notebook.*

## Objectifs pédagogiques

- Calculer et interpréter les indicateurs de tendance centrale et de dispersion (moyenne, médiane, écart-type)
- Comprendre pourquoi la médiane est souvent plus adaptée que la moyenne pour des données historiques
- Comparer deux groupes avec un test statistique (test t de Student)
- Mesurer une relation entre deux variables avec un coefficient de corrélation
- Tester l'indépendance entre deux variables catégorielles (test du χ², "khi-deux")
- Savoir formuler et interpréter une hypothèse statistique (H0 / H1, p-value) sans se noyer dans la théorie

**Principe général** : en sciences humaines, on ne fait presque jamais de statistiques "pour la statistique" — chaque calcul sert à répondre à une question historique précise. On gardera cette question en tête à chaque partie.

## Chargement des données

On repart du tableau nettoyé du cours précédent : une ligne par commune, avec sa population à différentes dates.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("DATA/pop_hist.csv", dtype=str)

# Nettoyage : population 2023 et population 1876 en nombres entiers
df["PMUN2023_num"] = df["PMUN2023"].str.replace(r"\s+", "", regex=True).astype(int)

# Attention : certaines communes (notamment en Corse) n'ont pas de valeur pour 1876.
# On les repère et on les met de côté avec pd.to_numeric(errors="coerce"), qui
# transforme en NaN (valeur manquante) tout ce qui ne peut pas être converti en nombre.
df["PTOT1876_num"] = pd.to_numeric(
    df["PTOT1876"].str.replace(r"\s+", "", regex=True), errors="coerce"
)
print(f"Communes sans donnée pour 1876 : {df['PTOT1876_num'].isna().sum()}")

print(f"{len(df)} communes.")
df[["LIBGEO", "REG", "DEP", "PTOT1876_num", "PMUN2023_num"]].head()

## Statistique descriptive : résumer un ensemble de valeurs

Avant de comparer ou de tester quoi que ce soit, on commence toujours par **décrire** ses données avec quelques nombres simples :

| Indicateur | Ce qu'il mesure | Fonction pandas |
|---|---|---|
| Moyenne | la valeur "centrale", sensible aux valeurs extrêmes | `.mean()` |
| Médiane | la valeur qui sépare l'échantillon en deux moitiés égales | `.median()` |
| Mode | la valeur la plus fréquente | `.mode()` |
| Écart-type | la dispersion autour de la moyenne | `.std()` |

### Un petit exemple pour comprendre la différence moyenne / médiane

Imaginons les revenus annuels (en milliers d'euros) de 6 habitants d'un village :

In [ ]:
revenus = pd.Series([18, 21, 19, 22, 20, 150])  # le dernier habitant est très riche

print("Moyenne :", revenus.mean())
print("Médiane :", revenus.median())

Une seule valeur extrême (150) fait bondir la moyenne à plus de 40, alors que 5 des 6 habitants gagnent autour de 20. La médiane, elle, reste stable à 20 : elle est **beaucoup moins sensible aux valeurs extrêmes**. En histoire comme en économie, on préfère donc souvent la médiane dès que les données sont susceptibles de contenir des cas exceptionnels.

### Appliquons cela à la population des communes en 2023

In [ ]:
pop = df["PMUN2023_num"]

print(f"Moyenne  : {pop.mean():,.0f} habitants")
print(f"Médiane  : {pop.median():,.0f} habitants")
print(f"Écart-type : {pop.std():,.0f} habitants")

La moyenne est presque **deux fois plus élevée** que la médiane ! Cela confirme ce qu'on a vu avec l'histogramme du cours précédent : une poignée de très grandes villes (Paris, Marseille, Lyon...) tire la moyenne vers le haut, alors que la commune "typique" est un petit village. Ici, **la médiane décrit mieux "la France des communes"** que la moyenne.

### Exercice 1

Calculez la moyenne, la médiane et l'écart-type de la population 2023 **uniquement pour les communes de votre département** (`DEP`, par exemple `"87"`). L'écart entre moyenne et médiane est-il aussi important que pour la France entière ? Pourquoi, à votre avis ?

In [ ]:
# Votre code ici

##  Regarder la distribution en entier

Les indicateurs des stats descriptives résument les données en un seul nombre, mais on perd beaucoup d'information. Un histogramme permet de voir **toute la distribution** d'un coup d'œil, et de repérer visuellement où se situent la moyenne et la médiane.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

tranches = np.logspace(1, 7, 30)
ax.hist(pop, bins=tranches, color="#4682b4", edgecolor="black")
ax.set_xscale("log")

# On ajoute des lignes verticales pour la moyenne et la médiane
ax.axvline(pop.mean(), color="firebrick", linestyle="--", label=f"Moyenne ({pop.mean():,.0f} hab.)")
ax.axvline(pop.median(), color="orange", linestyle="--", label=f"Médiane ({pop.median():,.0f} hab.)")

ax.set_title("Distribution de la population des communes françaises (2023)")
ax.set_xlabel("Population (échelle logarithmique)")
ax.set_ylabel("Nombre de communes")
ax.legend()

plt.show()

On voit bien que la moyenne (ligne rouge) est tirée vers la droite par la longue "traîne" des grandes villes, alors que la médiane (ligne orange) reste au cœur du gros de la distribution. On appelle ce type de distribution **asymétrique** ("skewed" en anglais). C'est très fréquent avec des données historiques ou sociales (revenus, tailles de ville, patrimoine, durée de vie des institutions...).

*Conséquence pratique : beaucoup de tests statistiques (comme le test t que l'on verra juste après) supposent que les données suivent une distribution à peu près symétrique ("normale"). Face à une distribution aussi asymétrique, une solution courante consiste à travailler sur le **logarithme** de la variable plutôt que sur la variable brute. C'est ce que l'on fera dans la suite.*

----

## Comparer deux groupes : le test t de Student

**Question historique** : les communes d'Île-de-France sont-elles, en moyenne, significativement plus peuplées que celles de Nouvelle-Aquitaine, ou est-ce que l'écart observé pourrait s'expliquer par le hasard de l'échantillon ?

C'est exactement le genre de question auquel répond un **test d'hypothèse**. La logique est toujours la même :

1. On formule une **hypothèse nulle H0** : "il n'y a pas de différence entre les deux groupes" ;
2. On formule une **hypothèse alternative H1** : "il y a une différence" ;
3. Le test renvoie une **p-value** : la probabilité d'observer un écart au moins aussi grand que celui mesuré, *si H0 était vraie* ;
4. Par convention, si la p-value est **inférieure à 0,05**, on considère l'écart comme "statistiquement significatif" et on rejette H0.

*La p-value ne dit jamais "il y a une différence importante" — elle dit seulement "cette différence a peu de chances d'être due au hasard".*

In [ ]:
pip install scipy

In [ ]:
from scipy import stats

# On travaille sur le logarithme de la population, pour la raison expliquée au dessus. 
# NB : quelques communes de la Meuse (les villages détruits de la bataille de
# Verdun, jamais repeuplés depuis 1916) ont une population officielle de 0 habitant.
# On les met de côté avec replace(0, np.nan), car le logarithme de 0 n'existe pas.
df["log_pop2023"] = np.log10(df["PMUN2023_num"].replace(0, np.nan))

idf = df[df["REG"] == "11"]["log_pop2023"].dropna()     # Île-de-France
naq = df[df["REG"] == "75"]["log_pop2023"].dropna()     # Nouvelle-Aquitaine

print(f"Île-de-France      : {len(idf)} communes, population médiane = {10**idf.median():,.0f} hab.")
print(f"Nouvelle-Aquitaine  : {len(naq)} communes, population médiane = {10**naq.median():,.0f} hab.")

t_stat, p_value = stats.ttest_ind(idf, naq, equal_var=False)
print(f"\nStatistique t = {t_stat:.2f}")
print(f"p-value       = {p_value:.2e}")

La p-value est extrêmement petite (bien en dessous de 0,05) : on rejette H0. L'écart de population entre communes d'Île-de-France et de Nouvelle-Aquitaine n'est pas dû au hasard : il est statistiquement très significatif. Cela confirme, chiffres à l'appui, l'intuition d'une organisation du territoire très différente entre les deux régions.

*Note méthodologique : le test t suppose que les données suivent une distribution à peu près normale. Si l'on travaillait directement sur la population brute (sans passer au logarithme), il serait plus prudent d'utiliser un test qui ne fait aucune hypothèse sur la forme de la distribution, comme le test de Mann-Whitney (`scipy.stats.mannwhitneyu`).*

### Exercice 2

Comparez la population médiane de **deux départements de votre choix** (colonne `DEP`) avec un test t. Formulez d'abord, en une phrase, l'hypothèse H0 et H1, puis interprétez la p-value obtenue.

In [ ]:
# Votre code ici

----

##  Mesurer une relation entre deux variables : la corrélation

**Question historique** : les communes qui étaient déjà grandes en 1876 sont-elles toujours les plus grandes en 2023 ? Autrement dit, la hiérarchie urbaine du 19e siècle s'est-elle perpétuée jusqu'à aujourd'hui ?

Le **coefficient de corrélation de Pearson** (`r`) mesure la force d'une relation linéaire entre deux variables numériques :

- `r` proche de **+1** : les deux variables augmentent ensemble ;
- `r` proche de **-1** : quand l'une augmente, l'autre diminue ;
- `r` proche de **0** : pas de relation linéaire.

In [ ]:
df["log_pop1876"] = np.log10(df["PTOT1876_num"].replace(0, np.nan))
df_corr = df.dropna(subset=["log_pop1876", "log_pop2023"])

r, p_value = stats.pearsonr(df_corr["log_pop1876"], df_corr["log_pop2023"])
print(f"Coefficient de corrélation r = {r:.2f}")
print(f"p-value                      = {p_value:.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(df_corr["log_pop1876"], df_corr["log_pop2023"], alpha=0.15, s=10, color="#4682b4")

ax.set_title(f"Population 1876 vs 2023 (r = {r:.2f})")
ax.set_xlabel("log10(Population 1876)")
ax.set_ylabel("log10(Population 2023)")

plt.show()

Le coefficient est élevé et positif : la taille d'une commune en 1876 est un **bon prédicteur** de sa taille aujourd'hui. La hiérarchie urbaine se transforme lentement. Il s'agit d'un résultat qui confirme, à l'échelle de tout le pays, une intuition bien connue des historiens de l'urbanisation.

*Attention : corrélation n'est pas causalité. Ce résultat ne dit pas "être grand en 1876 a causé la croissance jusqu'en 2023", seulement que les deux sont statistiquement liés.*

----

##  Tester l'indépendance de deux variables catégorielles : le test du χ²

Le test t et la corrélation fonctionnent avec des variables **numériques**. Mais en sciences humaines, on travaille très souvent avec des variables **catégorielles** (une religion, un statut social, un type de commune...). Pour tester si deux variables catégorielles sont liées, on utilise le **test du χ² d'indépendance**.

**Question historique** : la taille d'une commune dépend-elle de sa région, ou les catégories de taille sont-elles réparties au hasard entre les régions ?

On reprend la discrétisation du cours précédent (catégories de taille de ville), et on la croise avec la région, en adaptant les valeurs et labels de la discrétisation.

In [ ]:
df["taille_ville"] = pd.cut(
    df["PMUN2023_num"],
    bins=[0, 2000, 20000, 100_000, 200_000, float("inf")],
    labels=["Village", "Petite ville", "Ville moyenne", "Grande ville", "Très grande ville"],
)

print(df["taille_ville"].unique())

# On compare 3 régions contrastées
df_regions = df[df["REG"].isin(["11", "75", "84"])].copy()

df_regions["REG"] = df_regions["REG"].replace({
    "11": "Île-de-France",
    "75": "Nouvelle-Aquitaine",
    "84": "Auvergne-Rhône-Alpes",
})

tableau_croise = pd.crosstab(df_regions["REG"], df_regions["taille_ville"])
tableau_croise

In [ ]:
chi2, p_value, ddl, effectifs_attendus = stats.chi2_contingency(tableau_croise)

print(f"Statistique du χ² = {chi2:.1f}")
print(f"Degrés de liberté  = {ddl}")
print(f"p-value            = {p_value:.2e}")

La p-value est extrêmement faible (233 "0" après la virgule suivis de 504): on rejette l'hypothèse d'indépendance. La répartition des tailles de commune **dépend bien de la région**, ce qui n'a rien de surprenant vu le poids démographique de Paris dans le tableau croisé, mais le test permet de le confirmer de façon rigoureuse plutôt que de se fier à une impression visuelle.

###  Exercice 3 (synthèse)

Choisissez **une variable catégorielle de votre choix** à croiser avec `taille_ville` — par exemple en créant vos propres catégories de région, ou en comparant seulement deux départements plutôt que trois régions. Produisez le tableau croisé, lancez le test du χ², et rédigez une phrase de conclusion.

In [ ]:
# Votre code ici

## Pour conclure : quel outil pour quelle question ?

| Je veux... | J'utilise |
|---|---|
| Résumer une seule variable numérique | Moyenne, médiane, écart-type |
| Voir la forme d'une distribution | Histogramme |
| Comparer deux groupes sur une variable numérique | Test t de Student |
| Mesurer le lien entre deux variables numériques | Corrélation de Pearson |
| Tester le lien entre deux variables catégorielles | Test du χ² |

Dans tous les cas, la démarche reste la même : **poser une question historique claire avant de choisir l'outil statistique**, jamais l'inverse.